[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/capm_cost_of_capital.ipynb)

# CAPM and cost of capital full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (beta, the security market line, factor models, asset betas, the company cost of capital and
the after-tax WACC, a beta estimated from returns), **tier 3** stretch problems (combine ideas, spot the trap, run a
formula backwards) and **tier 4**, the case: Bangalore Cold Chain (a fictional company; our own data) can buy a
home-delivery business that should return 10 %, and has to decide what return that project must earn. Each tier is
harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same calculations in plain Python, with
`numpy` for the monthly returns and `pandas` for the case's sixty months, so you can run them on your own numbers.
And two cells you can move yourself: a beta that slides along the security market line, and a debt share that moves
a company's equity beta and its WACC while the business itself stays the same.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, move it and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Betas, R
squared and standard errors as plain numbers, to two decimals: `1.25`. Rates and returns as percentages: `12.85` for
12.85 % (typing the % sign is fine, and so is the rate itself, `0.1285`). Money as the question asks for it; the
case's values in $ millions, to two decimals: `34.78`. You can type the arithmetic too: `4 + 1.7*5.5`. For several
answers, separate them with commas, in the order the question lists them (the box's label shows the order). You can
also put your answer between the quotes in a cell's last line, `practice("P1.3", "")`, and run the cell: it checks
straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[CAPM and cost of capital cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/capm_cost_of_capital_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # cash in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# returns and cash CHALK · the market's part SKY · the rate investors ask VIOLET · debt MINT · a shortcut that misleads
# CORAL · the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own example · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log, "log": math.log}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…), exp(…), ln(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(dollars?|years?|months?|millions?|percent|points?)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol, pct=False):
    """Within the tolerance; a rate asked for in percent may also be typed as the rate itself (0.1285 for 12.85 %)."""
    if isinstance(got, str): return False
    tol = max(tol, 1e-7)
    return abs(got - want) <= tol or (pct and abs(got * 100 - want) <= tol)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Risk, beta and the market portfolio',
     "q": "A stock's returns have a standard deviation of 36 % and a correlation of 0.5 with the market, whose standard deviation is 18 %. What is the stock's beta?",
     "parts": ['beta'],
     "answer": [1.0], "tol": [0.005], "pct": [], "function": '—',
     "working": 'beta = correlation × sd of the stock / sd of the market = 0.5 × 36/18 = 1.0'},
    {"id": 'P1.2', "tier": 1, "group": 'Risk, beta and the market portfolio',
     "q": "Safe government bonds pay 3 %; the market is expected to return 8 % with a standard deviation of 16 %. You put 70 % in the market and 30 % in the safe bonds. What are the portfolio's expected return and standard deviation, in percent?",
     "parts": ['expected return (%)', 'standard deviation (%)'],
     "answer": [6.5, 11.2], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": 'return 3 + 0.7 × (8 − 3) = 6.5 %; sd 0.7 × 16 = 11.2 % (the safe bonds add no risk)'},
    {"id": 'P1.3', "tier": 1, "group": 'The capital asset pricing model',
     "q": 'The risk-free rate is 3 % and the market risk premium is 6 %. What return does the CAPM say investors expect from a stock with a beta of 1.5, in percent?',
     "parts": ['expected return (%)'],
     "answer": [12.0], "tol": [0.005], "pct": [0], "function": '—',
     "working": 'r = 3 + 1.5 × 6 = 12 %'},
    {"id": 'P1.4', "tier": 1, "group": 'The capital asset pricing model',
     "q": 'A portfolio holds 25 % in a stock with beta 2.0 and 75 % in a stock with beta 0.8. What is its beta?',
     "parts": ["the portfolio's beta"],
     "answer": [1.1], "tol": [0.005], "pct": [], "function": 'SUMPRODUCT',
     "working": '0.25 × 2.0 + 0.75 × 0.8 = 0.5 + 0.6 = 1.1. =SUMPRODUCT(weights, betas)'},
    {"id": 'P1.5', "tier": 1, "group": 'Tests and other models',
     "q": "In a three-factor model the risk-free rate is 4 % and the factor premiums are 5 % (market), 2 % (size) and 3 % (book-to-market). A stock's factor betas are 1.0, 0 and 1.0. What is its expected return, in percent?",
     "parts": ['expected return (%)'],
     "answer": [12.0], "tol": [0.005], "pct": [0], "function": 'SUMPRODUCT',
     "working": '4 + 1.0 × 5 + 0 × 2 + 1.0 × 3 = 12 %'},
    {"id": 'P1.6', "tier": 1, "group": 'Company and project cost of capital',
     "q": 'A company is financed 25 % by debt that investors expect to return 5 % and 75 % by equity expected to return 11 %. What is the company cost of capital, in percent?',
     "parts": ['company cost of capital (%)'],
     "answer": [9.5], "tol": [0.005], "pct": [0], "function": 'SUMPRODUCT',
     "working": '0.25 × 5 + 0.75 × 11 = 1.25 + 8.25 = 9.5 %'},
    {"id": 'P1.7', "tier": 1, "group": 'Company and project cost of capital',
     "q": 'A company is half debt and half equity by market value. Its debt beta is 0.2 and its equity beta is 1.6. What is its asset beta?',
     "parts": ['asset beta'],
     "answer": [0.9], "tol": [0.005], "pct": [], "function": 'SUMPRODUCT',
     "working": '0.5 × 0.2 + 0.5 × 1.6 = 0.9'},
    {"id": 'P1.8', "tier": 1, "group": 'Company and project cost of capital',
     "q": 'A company pays 6 % interest and its tax rate is 30 %. What does the debt cost after tax, in percent?',
     "parts": ['cost of debt after tax (%)'],
     "answer": [4.199999999999999], "tol": [0.005], "pct": [0], "function": '—',
     "working": '6 × (1 − 0.30) = 4.2 %'},
    {"id": 'P1.9', "tier": 1, "group": 'Estimating beta',
     "q": "Over five years of monthly returns, the covariance of a stock with the market is 0.0018 and the market's variance is 0.0015. What is the stock's beta?",
     "parts": ['beta'],
     "answer": [1.2], "tol": [0.005], "pct": [], "function": 'SLOPE',
     "working": 'beta = covariance / variance of the market = 0.0018/0.0015 = 1.2. =COVARIANCE.S(stock, market)/VAR.S(market), or =SLOPE(stock, market)'},
    {"id": 'P1.10', "tier": 1, "group": 'Project risk and certainty equivalents',
     "q": 'A project is expected to pay $1,090 in one year. Its cost of capital is 9 % and the risk-free rate is 4 %. What certain amount in one year is worth the same today?',
     "parts": ['the certain amount ($)'],
     "answer": [1040.0], "tol": [0.005], "pct": [], "function": '—',
     "working": 'PV = 1,090/1.09 = 1,000; the certain amount that is worth 1,000 today is 1,000 × 1.04 = 1,040 (= 1,090 × 1.04/1.09)'},
    {"id": 'P2.1', "tier": 2, "group": 'Risk, beta and the market portfolio',
     "q": 'A mining stock has a standard deviation of 50 % and a correlation of 0.3 with the market (standard deviation 20 %). Find its beta, the part of its standard deviation that is market risk (percent), and the share of its variance that is market risk (percent).',
     "parts": ['beta', 'the market part of its standard deviation (%)', "market risk's share of its variance (%)"],
     "answer": [0.7499999999999999, 14.999999999999998, 9.0], "tol": [0.005, 0.005, 0.005], "pct": [1, 2], "function": '—',
     "working": 'beta = 0.3 × 50/20 = 0.75; market part = 0.75 × 20 = 15 % of the 50 %; share of variance = 0.3² = 9 %: the other 91 % diversifies away'},
    {"id": 'P2.2', "tier": 2, "group": 'Risk, beta and the market portfolio',
     "q": 'Safe government bonds pay 4 %; the market is expected to return 9 % with a standard deviation of 20 %. You want an expected return of 10 %. What weight in the market do you need, and what standard deviation do you accept (percent)?',
     "parts": ['weight in the market (as a number: 0.5 for 50 %)', 'standard deviation (%)'],
     "answer": [1.2, 24.0], "tol": [0.005, 0.005], "pct": [1], "function": '—',
     "working": '10 = 4 + w × 5 → w = 1.2: borrow 20 % of your money and put 120 % in the market; sd = 1.2 × 20 = 24 %'},
    {"id": 'P2.3', "tier": 2, "group": 'The capital asset pricing model',
     "q": 'The risk-free rate is 3.5 % and the market risk premium is 5.5 %. What does the CAPM give for a stock with beta 0.6, and for one with beta 1.7 (percent)?',
     "parts": ['expected return at beta 0.6 (%)', 'expected return at beta 1.7 (%)'],
     "answer": [6.800000000000001, 12.85], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": '3.5 + 0.6 × 5.5 = 6.8 %; 3.5 + 1.7 × 5.5 = 12.85 %'},
    {"id": 'P2.4', "tier": 2, "group": 'The capital asset pricing model',
     "q": 'A stock with a beta of 1.3 is priced to return 12 %. The risk-free rate is 4 % and the market risk premium 5 %. By how many percentage points does it sit above or below the security market line, and what happens to its price?',
     "parts": ['points above the line (percentage points; a minus sign if below)'],
     "answer": [1.4999999999999987], "tol": [0.005], "pct": [], "function": '—',
     "working": 'the line asks 4 + 1.3 × 5 = 10.5 %; the stock offers 12 %, 1.5 points above the line: investors buy, the price rises, and the expected return falls back to 10.5 %'},
    {"id": 'P2.5', "tier": 2, "group": 'Tests and other models',
     "q": 'Risk-free 4 %; factor premiums 5 % (market), 2 % (size), 3 % (book-to-market). A large value firm has factor betas 1.2, −0.3 and 0.5. Find its expected return with three factors, and with the market alone (percent).',
     "parts": ['with three factors (%)', 'with the market alone (%)'],
     "answer": [10.900000000000002, 10.0], "tol": [0.005, 0.005], "pct": [0, 1], "function": 'SUMPRODUCT',
     "working": '4 + 1.2 × 5 − 0.3 × 2 + 0.5 × 3 = 4 + 6 − 0.6 + 1.5 = 10.9 %; with the market alone 4 + 6 = 10 %'},
    {"id": 'P2.6', "tier": 2, "group": 'Company and project cost of capital',
     "q": 'A company has debt worth $150 million (beta 0.15) and equity worth $450 million (beta 1.3). The risk-free rate is 4 % and the market risk premium 5 %. Find its asset beta (four decimals) and its company cost of capital (percent).',
     "parts": ['asset beta (four decimals)', 'company cost of capital (%)'],
     "answer": [1.0125, 9.062500000000002], "tol": [5e-05, 0.005], "pct": [1], "function": 'SUMPRODUCT',
     "working": 'weights 25 % and 75 %: 0.25 × 0.15 + 0.75 × 1.3 = 1.0125; 4 + 1.0125 × 5 = 9.06 %'},
    {"id": 'P2.7', "tier": 2, "group": 'Company and project cost of capital',
     "q": 'The same company pays tax at 25 %. Find its cost of debt, its cost of equity and its after-tax weighted-average cost of capital (percent).',
     "parts": ['cost of debt (%)', 'cost of equity (%)', 'after-tax WACC (%)'],
     "answer": [4.75, 10.500000000000002, 8.765625], "tol": [0.005, 0.005, 0.005], "pct": [0, 1, 2], "function": '—',
     "working": 'debt 4 + 0.15 × 5 = 4.75 %; equity 4 + 1.3 × 5 = 10.5 %; WACC = 0.25 × 4.75 × 0.75 + 0.75 × 10.5 = 0.89 + 7.875 = 8.77 %'},
    {"id": 'P2.8', "tier": 2, "group": 'Estimating beta',
     "q": "Five months of returns, in percent. Market: 5, −1, 1, 3, −3. Stock: 5, 1, 1, 7, −4. Find the stock's beta and R squared by hand, then check with SLOPE and RSQ.",
     "parts": ['beta', 'R squared'],
     "answer": [1.2, 0.7999999999999999], "tol": [0.005, 0.005], "pct": [], "function": 'SLOPE, RSQ',
     "working": 'means 1 and 2; sum of squared market deviations 40; sum of products 48; beta = 48/40 = 1.2000; R squared = beta² × 40/72 = 0.8000. =SLOPE(stock, market), =RSQ(stock, market)'},
    {"id": 'P2.9', "tier": 2, "group": 'Estimating beta',
     "q": "A regression of a stock's monthly returns on the market's gives STEYX = 4.0 (percent a month), and DEVSQ of the market's returns is 400. What is the standard error of the beta?",
     "parts": ['standard error of the beta'],
     "answer": [0.2], "tol": [0.005], "pct": [], "function": 'STEYX, DEVSQ, SQRT',
     "working": 'standard error = STEYX/√DEVSQ = 4/20 = 0.2. =STEYX(stock, market)/SQRT(DEVSQ(market))'},
    {"id": 'P2.10', "tier": 2, "group": 'Project risk and certainty equivalents',
     "q": "A plant's revenues have a present value of $500 million and a beta of 0.4. Its fixed costs have a present value of $150 million and its variable costs $250 million. What is the beta of the plant itself?",
     "parts": ["the plant's asset beta"],
     "answer": [1.0], "tol": [0.005], "pct": [], "function": '—',
     "working": 'the assets are worth 500 − 150 − 250 = 100; asset beta = 0.4 × (1 + 150/100) = 1.0'},
    {"id": 'P3.1', "tier": 3, "group": 'The capital asset pricing model',
     "q": "A drug developer's shares have a standard deviation of 70 % but a correlation of only 0.1 with the market. A power utility's have a standard deviation of 25 % and a correlation of 0.6. The market's standard deviation is 20 %, the risk-free rate 4 %, the premium 5 %. Which stock must offer the higher expected return? Give both (percent).",
     "parts": ['the drug developer (%)', 'the power utility (%)'],
     "answer": [5.75, 7.75], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": 'betas 0.1 × 70/20 = 0.35 and 0.6 × 25/20 = 0.75: the developer 5.75 %, the utility 7.75 %. The far more volatile stock earns less: most of its risk diversifies away, and only market risk earns a reward'},
    {"id": 'P3.2', "tier": 3, "group": 'The capital asset pricing model',
     "q": 'Two stocks sit on the security market line: beta 0.8 with an expected return of 8.2 %, and beta 1.4 with 11.8 %. What are the market risk premium and the risk-free rate (percent)?',
     "parts": ['market risk premium (%)', 'risk-free rate (%)'],
     "answer": [6.0, 3.4], "tol": [0.005, 0.005], "pct": [0, 1], "function": 'SLOPE, INTERCEPT',
     "working": 'the slope is the premium: (11.8 − 8.2)/(1.4 − 0.8) = 6 %; the risk-free rate is the height at beta zero: 8.2 − 0.8 × 6 = 3.4 %'},
    {"id": 'P3.3', "tier": 3, "group": 'Tests and other models',
     "q": 'Risk-free 4 %; factor premiums 5 %, 2 % and 3 %. A stock with a market beta of 0.9 and a size beta of 0.5 is expected to return 11 %. What book-to-market beta does that imply?',
     "parts": ['book-to-market beta'],
     "answer": [0.5000000000000001], "tol": [0.005], "pct": [], "function": '—',
     "working": '11 − 4 = 7 points of premium; the market gives 4.5 and size 1.0; the remaining 1.5 = beta × 3 → 0.5'},
    {"id": 'P3.4', "tier": 3, "group": 'Company and project cost of capital',
     "q": "A listed firm in the business you want to enter has an equity beta of 1.4, with debt 30 % of its value (debt beta 0.2). Find the business's asset beta. You would finance your project half with debt (debt beta 0.3). Find your equity beta, and the project's cost of capital (risk-free 4 %, premium 5 %; percent).",
     "parts": ["the business's asset beta", 'your equity beta', "the project's cost of capital (%)"],
     "answer": [1.04, 1.78, 9.2], "tol": [0.005, 0.005, 0.005], "pct": [2], "function": '—',
     "working": "unlever: 0.3 × 0.2 + 0.7 × 1.4 = 1.04; relever: 1.04 + (1.04 − 0.3) × 50/50 = 1.78; the project's cost of capital stays 4 + 1.04 × 5 = 9.2 %: it depends on the use of the money, not on how the project is financed"},
    {"id": 'P3.5', "tier": 3, "group": 'Company and project cost of capital',
     "q": "A company's cost of capital is 9 %. Project one has an asset beta of 0.5 and should return 7.5 %; project two has an asset beta of 1.8 and should return 11 %. Risk-free 4 %, premium 5 %. What return does each project need (percent), and which should the company take?",
     "parts": ['project one needs (%)', 'project two needs (%)'],
     "answer": [6.5, 13.0], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": 'project one needs 4 + 0.5 × 5 = 6.5 % and offers 7.5: take it. Project two needs 4 + 1.8 × 5 = 13 % and offers 11: leave it. The 9 % company hurdle gets both wrong'},
    {"id": 'P3.6', "tier": 3, "group": 'Estimating beta',
     "q": "A stock's monthly returns have a standard deviation of 9 %, the market's 5 %, and the regression's R squared is 0.49 (the stock rises with the market). Find the beta, and the standard deviation of the stock's specific risk (percent a month).",
     "parts": ['beta', 'specific standard deviation (percent a month)'],
     "answer": [1.26, 6.427285585688565], "tol": [0.005, 0.005], "pct": [], "function": 'SQRT',
     "working": 'correlation = √0.49 = 0.7; beta = 0.7 × 9/5 = 1.26; specific variance = 81 × 0.51 = 41.31, so the specific sd is 6.43 %'},
    {"id": 'P3.7', "tier": 3, "group": 'Estimating beta',
     "q": 'A regression gives a beta of 0.9 with a standard error of 0.3. Give the range two standard errors either side, and the cost of equity at each end (risk-free 4 %, premium 5 %; percent). Then: if 16 similar firms each have that standard error and their errors are unrelated, what is the standard error of their average beta?',
     "parts": ['low end of the range', 'high end of the range', 'cost of equity at the low end (%)', 'cost of equity at the high end (%)', 'standard error of the average beta (three decimals)'],
     "answer": [0.3, 1.5, 5.5, 11.500000000000002, 0.075], "tol": [0.005, 0.005, 0.005, 0.005, 0.005], "pct": [2, 3], "function": 'SQRT',
     "working": "0.9 ± 0.6 → 0.3 to 1.5; cost of equity 5.5 % to 11.5 %: one firm's beta says little. The average of 16: 0.3/√16 = 0.075, four times tighter"},
    {"id": 'P3.8', "tier": 3, "group": 'Project risk and certainty equivalents',
     "q": "A project should pay $2,000,000 in three years. Its cost of capital is 12 % and the risk-free rate 4 %. Find its present value ($ millions, four decimals) and the year-three certainty equivalent ($ millions, four decimals). Then suppose all the risk is settled in year one, after which the cash is certain: take one year's risk deduction only, and find the present value ($ millions, four decimals).",
     "parts": ['present value ($ millions, four decimals)', 'year-three certainty equivalent ($ millions, four decimals)', 'present value if the risk is settled in year one ($ millions, four decimals)'],
     "answer": [1.4235604956268217, 1.6013119533527693, 1.6509932375316987], "tol": [5e-05, 5e-05, 5e-05], "pct": [], "function": '—',
     "working": 'PV = 2/1.12³ = 1.4236; certainty equivalent = 2 × (1.04/1.12)³ = 1.6013 (and 1.6013/1.04³ gives the same PV). Risk settled in year one: 2 × (1.04/1.12) = 1.8571, discounted three years at 4 %: 1.6510. One 12 % rate for every year assumes the risk keeps building, and undervalues this project'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Fit the line through the sixty months (share returns against market returns). Give the slope (the equity beta) and the intercept (percent a month).',
     "parts": ['slope: the equity beta', 'intercept (percent a month)'],
     "answer": [1.1999675255370585, 0.0998910171014622], "tol": [0.005, 0.005], "pct": [], "function": 'SLOPE, INTERCEPT',
     "working": '=SLOPE(share, market) = 1.2000; =INTERCEPT(share, market) = 0.0999. The returns are built so the slope is 1.20'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Find R squared and the correlation. Then get beta a second way: correlation times the share's standard deviation over the market's.",
     "parts": ['R squared', 'correlation', 'beta the second way'],
     "answer": [0.3599618448843461, 0.5999682032277596, 1.1999675255370585], "tol": [0.005, 0.005, 0.005], "pct": [], "function": 'RSQ, CORREL, STDEV.S',
     "working": "=RSQ = 0.3600: 36 % of the shares' variance is market risk; =CORREL = 0.6000; sd 11.55 % and 5.77 % a month; 0.60 × 2.00 = 1.20, the slope again"},
    {"id": 'P4.3', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'How sure is the beta? Give STEYX (percent a month), the standard error of the beta, and the range two standard errors either side (use 2.0017 for 58 degrees of freedom).',
     "parts": ['STEYX (percent a month)', 'standard error of the beta', 'low end of the range', 'high end of the range'],
     "answer": [9.317045066620596, 0.2101020036293217, 0.7794063448722452, 1.6205287062018718], "tol": [0.005, 0.005, 0.005, 0.005], "pct": [], "function": 'STEYX, DEVSQ, SQRT',
     "working": '=STEYX = 9.32; standard error = STEYX/SQRT(DEVSQ(market)) = 0.2101; 1.20 ± 2.0017 × 0.21 → 0.78 to 1.62'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'With the risk-free rate at 4 % and the market risk premium at 5 %, find the cost of equity at a beta of 1.2, and at each end of the range from P4.3 (percent).',
     "parts": ['cost of equity at a beta of 1.2 (%)', 'at the low end (%)', 'at the high end (%)'],
     "answer": [10.0, 7.897031724361227, 12.10264353100936], "tol": [0.005, 0.005, 0.005], "pct": [0, 1, 2], "function": '—',
     "working": "4 + 1.2 × 5 = 10 %; at the ends 7.90 % and 12.10 %: about two points either side, which is why analysts also look at an industry's average beta"},
    {"id": 'P4.5', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'The company has debt worth $80 million (beta 0.2) and 32 million shares at $10. Find the cost of debt (percent), the asset beta, and the company cost of capital (percent).',
     "parts": ['cost of debt (%)', 'asset beta', 'company cost of capital (%)'],
     "answer": [5.0, 1.0, 9.0], "tol": [0.005, 0.005, 0.005], "pct": [0, 2], "function": 'SUMPRODUCT',
     "working": 'debt 4 + 0.2 × 5 = 5 %; weights 80/400 = 20 % and 80 %; asset beta 0.2 × 0.2 + 0.8 × 1.2 = 1.0; company cost of capital 0.2 × 5 + 0.8 × 10 = 9 % (and 4 + 1.0 × 5 = 9 %)'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'The tax rate is 25 %. Find the after-tax weighted-average cost of capital (percent). What would it be at a tax rate of 30 %?',
     "parts": ['after-tax WACC (%)', 'at a 30 % tax rate (%)'],
     "answer": [8.75, 8.7], "tol": [0.005, 0.005], "pct": [0, 1], "function": '—',
     "working": '0.2 × 5 × 0.75 + 0.8 × 10 = 0.75 + 8 = 8.75 %; at 30 %: 0.2 × 5 × 0.70 + 8 = 8.70 %'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "The delivery project: $40 million for an expected $4 million a year, for good. Unlever the three comparable firms (equity betas 1.8, 1.7, 2.0; debt 25 %, 10 %, 25 % of value; debt beta 0.2) and average. Give the project's asset beta, its cost of capital (percent), and its NPV at 9 %, at 8.75 % and at its own rate ($ millions).",
     "parts": ["the project's asset beta", 'its cost of capital (%)', 'NPV at 9 % ($ millions)', 'NPV at 8.75 % ($ millions)', 'NPV at its own rate ($ millions)'],
     "answer": [1.5, 11.500000000000002, 4.444444444444443, 5.714285714285715, -5.217391304347835], "tol": [0.005, 0.005, 0.005, 0.005, 0.005], "pct": [1], "function": 'SUMPRODUCT, AVERAGE',
     "working": 'asset betas 1.40, 1.55, 1.55: average 1.50; 4 + 1.5 × 5 = 11.5 %; NPV = 4/0.09 − 40 = +4.44; 4/0.0875 − 40 = +5.71; 4/0.115 − 40 = -5.22. (Averaging the equity betas without unlevering would give 1.83 and 13.17 %: too high.)'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Write the recommendation in two sentences, with the numbers that anchor it: the asset beta at which the project would break even, the most the business is worth paying for ($ millions), and the yearly cash the $40 million would need ($ millions).',
     "parts": ['break-even asset beta', 'the most the business is worth ($ millions)', 'the yearly cash the $40 million would need ($ millions)'],
     "answer": [1.2, 34.782608695652165, 4.6000000000000005], "tol": [0.005, 0.005, 0.005], "pct": [], "function": '—',
     "working": 'Do not invest at $40 million: at the 11.5 % its risk calls for, the business is worth 34.78 million, 5.22 less than it costs. It would break even only with an asset beta of 1.2 (all three comparable firms are above that) or with 4.6 million of cash a year'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t, i in q["pct"]) for i, (g, w, t) in enumerate(zip(got, q["answer"], q["tol"]))]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't; a rate may be typed as a percentage or as itself
assert right(read_part("3 + 1.5*6"), answers("P1.3")[0], 0.005) and not right(read_part("12.5"), answers("P1.3")[0], 0.005)
assert right(read_part("12%"), 12.0, 0.005, True) and right(read_part("0.12"), 12.0, 0.005, True)
assert not right(read_part("0.12"), 12.0, 0.005) and not right(read_part("twelve"), 12.0, 0.005, True)

## Tier 1 · Warm-up

One idea, one step, small numbers: a beta from a correlation, a mix of the market and safe bonds, the CAPM, a
portfolio's beta, a three-factor return, a company cost of capital, an asset beta, debt after tax, a beta from a
covariance, and a certainty equivalent.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one line. The cell below writes them once (rates as fractions: 0.04 for 4 %), then
computes every warm-up answer and checks it against the practice set. `weighted` is the spreadsheet's SUMPRODUCT: a
portfolio's beta, a factor model's premium and a company's cost of capital are all weighted sums.

In [ ]:
def capm(beta, rf, premium): return rf + beta * premium                    # the security market line
def beta_from_corr(corr, sd, sd_market): return corr * sd / sd_market      # correlation × sd of the stock ÷ sd of the market
def mix(w, rf, rm, sd_market):                                             # w in the market, 1 − w in safe bonds
    return rf + w * (rm - rf), abs(w) * sd_market                         # (expected return, standard deviation)
def weighted(weights, values): return float(np.dot(weights, values))       # SUMPRODUCT(weights, values)
def factor_return(rf, betas, premiums): return rf + weighted(betas, premiums)
def asset_beta(beta_debt, beta_equity, debt, equity):                      # unlever: the weighted average
    return weighted([debt, equity], [beta_debt, beta_equity]) / (debt + equity)
def equity_beta(beta_assets, beta_debt, debt, equity):                     # relever: the same line, solved for the equity
    return beta_assets + (beta_assets - beta_debt) * debt / equity
def company_cost(r_debt, r_equity, debt, equity):                          # the return on the debt and equity together
    return weighted([debt, equity], [r_debt, r_equity]) / (debt + equity)
def wacc(r_debt, r_equity, debt, equity, tax):                             # the same, with the debt's cost after tax
    return weighted([debt, equity], [r_debt * (1 - tax), r_equity]) / (debt + equity)
def certainty_equivalent(cash, r, rf, t=1): return cash * ((1 + rf) / (1 + r)) ** t

b11 = beta_from_corr(0.5, 0.36, 0.18)                                      # P1.1
ret12, sd12 = mix(0.7, 0.03, 0.08, 0.16)                                   # P1.2
r13 = capm(1.5, 0.03, 0.06)                                                # P1.3
b14 = weighted([0.25, 0.75], [2.0, 0.8])                                   # P1.4: a portfolio's beta
r15 = factor_return(0.04, [1.0, 0.0, 1.0], [0.05, 0.02, 0.03])             # P1.5: three factors
r16 = company_cost(0.05, 0.11, 25, 75)                                     # P1.6
b17 = asset_beta(0.2, 1.6, 50, 50)                                         # P1.7
after_tax = 0.06 * (1 - 0.30)                                              # P1.8
b19 = 0.0018 / 0.0015                                                      # P1.9: covariance ÷ the market's variance
ce = certainty_equivalent(1090, 0.09, 0.04)                                # P1.10

print(f"beta {b11:.2f}   mix: return {ret12:.2%}, sd {sd12:.2%}   CAPM {r13:.2%}   portfolio beta {b14:.2f}   "
      f"three factors {r15:.2%}")
print(f"company cost of capital {r16:.2%}   asset beta {b17:.2f}   debt after tax {after_tax:.2%}   beta {b19:.2f}   "
      f"certainty equivalent {ce:,.2f}")
for k, v in (("P1.1", [b11]), ("P1.2", [100 * ret12, 100 * sd12]), ("P1.3", [100 * r13]), ("P1.4", [b14]),
             ("P1.5", [100 * r15]), ("P1.6", [100 * r16]), ("P1.7", [b17]), ("P1.8", [100 * after_tax]), ("P1.9", [b19]),
             ("P1.10", [ce])):
    assert agrees(k, v), k

### The six months, in four lines

The course's six months of returns, in percent: the market's, and a furniture maker's shares. `np.polyfit` fits the
straight line through the six points; its slope is the beta the video worked out by hand, 1.5, and its intercept is
one half. The correlation squared is R squared, 0.75. The stock goes second in `polyfit` (x first, then y): the
opposite order from a spreadsheet's SLOPE, which takes the stock first.

In [ ]:
market = np.array([3, -3, 7, -1, -2, 2])
stock = np.array([7, -6, 12, 0, 1, -2])
beta, intercept = np.polyfit(market, stock, 1)
r_squared = np.corrcoef(market, stock)[0, 1] ** 2
print(f"beta = {beta:.2f}   intercept = {intercept:.2f}   R squared = {r_squared:.2f}")

In [ ]:
# The hand method, line by line: beta is the covariance over the market's variance, and its standard error is the
# typical miss over the square root of the market's squared gaps.
covariance = np.cov(stock, market)[0, 1]                                   # divides by months − 1, like COVARIANCE.S
variance = np.var(market, ddof=1)                                          # VAR.S
miss = stock - (intercept + beta * market)                                 # how far each month sits from the line
typical_miss = math.sqrt((miss ** 2).sum() / (len(market) - 2))            # STEYX
squared_gaps = ((market - market.mean()) ** 2).sum()                       # DEVSQ(market)
standard_error = typical_miss / math.sqrt(squared_gaps)                    # STEYX/SQRT(DEVSQ(market))
print(f"covariance {covariance:.1f} / variance {variance:.1f} = beta {covariance / variance:.2f}   "
      f"typical miss {typical_miss:.2f}   standard error of beta {standard_error:.2f}")
assert round(float(beta), 9) == 1.5 and round(float(intercept), 9) == 0.5 and round(float(r_squared), 9) == 0.75   # the video's numbers
assert round(float(covariance), 9) == 21 and round(float(variance), 9) == 14 and round(float(squared_gaps), 9) == 70
assert round(float((miss ** 2).sum()), 9) == 52.5 and round(typical_miss, 2) == 3.62 and round(standard_error, 2) == 0.43

### Slide beta along the security market line

The line starts at the risk-free rate, 4 %, where beta is zero, and climbs by the market risk premium, 5 points, for
each unit of beta. At a beta of 1 it passes through the market itself, 9 %. Slide beta: a power utility at 0.5 is
asked for 6.5 %, an airline at 1.8 for 13 %. Nothing else about a stock moves its place on the line: only market
risk earns a reward.

In [ ]:
def line_picture(beta=1.8):
    rf, premium = 0.04, 0.05
    r = capm(beta, rf, premium)
    betas = np.linspace(0, 2.5, 50)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(betas, 100 * capm(betas, rf, premium), color=VIOLET)
    ax.plot([beta, beta, 0], [0, 100 * r, 100 * r], color=DIM, lw=1.2, ls="--")
    ax.scatter([0, 1], [100 * rf, 100 * capm(1, rf, premium)], s=60, color=CHALK, zorder=3)
    ax.annotate("safe bonds", (0, 100 * rf), textcoords="offset points", xytext=(10, -16), color=CHALK, fontsize=12)
    ax.annotate("the market", (1, 100 * capm(1, rf, premium)), textcoords="offset points", xytext=(8, -18), color=CHALK,
                fontsize=12)
    ax.scatter([beta], [100 * r], s=120, color=GOLD, zorder=4)
    ax.set_xlim(0, 2.5); ax.set_ylim(0, 18); ax.set_xlabel("beta"); ax.set_ylabel("expected return (%)")
    fig.suptitle(f"beta {beta:g}: 4 % + {beta:g} × 5 % = {100 * r:.2f} %", x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert [round(100 * capm(b, 0.04, 0.05), 9) for b in (0, 0.5, 1, 1.8)] == [4, 6.5, 9, 13]         # the video's numbers

In [ ]:
interact(line_picture, beta=FloatSlider(value=1.8, min=0, max=2.5, step=0.05, description="beta"));

## Tier 2 · Core

The module's standard problems, several steps each: how much of a stock's risk is market risk, a mix that reaches a
target return, two stocks on the line, a stock priced off the line, three factors against one, an asset beta and a
company cost of capital, the after-tax WACC, a beta and R squared from five months of returns, a standard error, and
operating leverage.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Estimating a beta** is the four lines above wrapped in one function: `fit_line(market, stock)` returns the beta,
the intercept, R squared, the typical miss and the standard error of beta (a spreadsheet's SLOPE, INTERCEPT, RSQ,
STEYX and STEYX/SQRT(DEVSQ(market))). **The company** uses `asset_beta`, `company_cost` and `wacc` from tier 1. Each
line below computes one core answer and checks it against the practice set.

In [ ]:
def fit_line(market, stock):
    """The least-squares line of a stock's returns on the market's: (beta, intercept, R squared, typical miss,
    standard error of beta)."""
    x, y = np.asarray(market, float), np.asarray(stock, float)
    slope, icept = np.polyfit(x, y, 1)
    miss = y - (icept + slope * x)
    typical = math.sqrt((miss ** 2).sum() / (len(x) - 2))
    return (float(slope), float(icept), float(np.corrcoef(x, y)[0, 1] ** 2), typical,
            typical / math.sqrt(((x - x.mean()) ** 2).sum()))

b21 = beta_from_corr(0.3, 0.50, 0.20)                                      # P2.1: the market part is beta × the market's sd
w22 = (0.10 - 0.04) / (0.09 - 0.04)                                        # P2.2: the weight that reaches 10 %
r23 = [capm(b, 0.035, 0.055) for b in (0.6, 1.7)]                          # P2.3
gap24 = 0.12 - capm(1.3, 0.04, 0.05)                                       # P2.4: above the line if positive
three, one = factor_return(0.04, [1.2, -0.3, 0.5], [0.05, 0.02, 0.03]), capm(1.2, 0.04, 0.05)     # P2.5
ba26 = asset_beta(0.15, 1.3, 150, 450)                                     # P2.6
rd27, re27 = capm(0.15, 0.04, 0.05), capm(1.3, 0.04, 0.05)                 # P2.7
wacc27 = wacc(rd27, re27, 150, 450, 0.25)
b28, _, rsq28, _, _ = fit_line([5, -1, 1, 3, -3], [5, 1, 1, 7, -4])        # P2.8: SLOPE and RSQ
se29 = 4.0 / math.sqrt(400)                                                # P2.9: STEYX ÷ √DEVSQ
plant = 0.4 * (1 + 150 / (500 - 150 - 250))                                # P2.10: revenue beta × (1 + fixed costs ÷ assets)

print(f"beta {b21:.2f}: market part {b21 * 0.20:.0%} of the 50 %, {0.3 ** 2:.0%} of the variance   "
      f"weight {w22:.2f}, sd {mix(w22, 0.04, 0.09, 0.20)[1]:.0%}")
print(f"CAPM {r23[0]:.2%} and {r23[1]:.2%}   {100 * gap24:+.1f} points off the line   three factors {three:.2%}, "
      f"the market alone {one:.2%}")
print(f"asset beta {ba26:.4f}, company cost of capital {capm(ba26, 0.04, 0.05):.4%}   debt {rd27:.2%}, equity "
      f"{re27:.2%}, WACC {wacc27:.4%}")
print(f"five months: beta {b28:.2f}, R squared {rsq28:.2f}   standard error {se29:.2f}   the plant's beta {plant:.2f}")
for k, v in (("P2.1", [b21, 100 * b21 * 0.20, 100 * 0.3 ** 2]), ("P2.2", [w22, 100 * mix(w22, 0.04, 0.09, 0.20)[1]]),
             ("P2.3", [100 * r for r in r23]), ("P2.4", [100 * gap24]), ("P2.5", [100 * three, 100 * one]),
             ("P2.6", [ba26, 100 * capm(ba26, 0.04, 0.05)]), ("P2.7", [100 * rd27, 100 * re27, 100 * wacc27]),
             ("P2.8", [b28, rsq28]), ("P2.9", [se29]), ("P2.10", [plant])):
    assert agrees(k, v), k
assert [round(v, 9) for v in fit_line(market, stock)[:3]] == [1.5, 0.5, 0.75]                      # the six months again

### Slide the debt share

The course's firm: assets with a beta of 1.24, financed 30 % by debt (beta 0.4) and 70 % by equity; risk-free 4 %,
premium 5 %, tax 25 %. Slide the debt share and the business doesn't change, so the asset beta and the company cost
of capital (VIOLET) stay where they are, at 10.2 %. The equity beta and the cost of equity climb: 1.6 and 12 % at
30 % debt, 2.08 at 50 %, because the shareholders carry the same business risk on a smaller stake. The after-tax
WACC (SKY) slips below the company cost of capital as debt rises, because interest is paid before tax. To keep the
picture simple the debt's beta is held at 0.4; a firm that borrows much more would find its debt riskier too.

In [ ]:
def firm_at(debt_share, beta_assets=1.24, beta_debt=0.4, rf=0.04, premium=0.05, tax=0.25):
    """(equity beta, cost of equity, company cost of capital, after-tax WACC) with this share of debt in the value."""
    be = equity_beta(beta_assets, beta_debt, debt_share, 1 - debt_share)
    r_d, r_e = capm(beta_debt, rf, premium), capm(be, rf, premium)
    return (be, r_e, company_cost(r_d, r_e, debt_share, 1 - debt_share), wacc(r_d, r_e, debt_share, 1 - debt_share, tax))

def debt_picture(debt_pct=30.0):
    shares = np.linspace(0, 0.8, 81)
    curve = np.array([firm_at(d) for d in shares])
    be, r_e, r_a, w = firm_at(debt_pct / 100)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * shares, 100 * curve[:, 1], color=CHALK, label="cost of equity")
    ax.plot(100 * shares, 100 * curve[:, 2], color=VIOLET, label="company cost of capital")
    ax.plot(100 * shares, 100 * curve[:, 3], color=SKY, label="after-tax WACC")
    ax.axhline(100 * capm(0.4, 0.04, 0.05), color=MINT, lw=1.6, ls="--", label="cost of debt")
    ax.axvline(debt_pct, color=DIM, lw=1.2, ls="--")
    ax.scatter([debt_pct] * 3, [100 * r_e, 100 * r_a, 100 * w], s=90, color=GOLD, zorder=4)
    ax.set_xlim(0, 80); ax.set_ylim(4, 32); ax.set_xlabel("debt as a share of the firm's value (%)")
    ax.set_ylabel("expected return (%)"); ax.legend(loc="upper left", frameon=False)
    fig.suptitle(f"debt {debt_pct:g} %: equity beta {be:.2f}, cost of equity {100 * r_e:.2f} %, company cost of capital "
                 f"{100 * r_a:.2f} %, WACC {100 * w:.2f} %", x=0.08, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

assert [round(v, 9) for v in firm_at(0.30)] == [1.6, 0.12, 0.102, 0.0975]                          # the video's numbers
assert round(firm_at(0.50)[0], 9) == 2.08 and round(firm_at(0.0)[0], 9) == 1.24 and round(firm_at(0.0)[3], 9) == 0.102

In [ ]:
interact(debt_picture, debt_pct=FloatSlider(value=30, min=0, max=80, step=1, description="debt (%)"));

## Tier 3 · Stretch

Combine ideas, spot the trap, run a formula backwards: a volatile stock that earns less than a calm one, the line
through two stocks, a factor beta from a return, unlevering one firm's beta and relevering it for your own project,
two projects against one company hurdle, a beta from R squared, how far to trust one estimate, and a project whose
risk is settled early.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**Running a formula backwards** is a line of algebra each time: the premium is the slope between two points on the
line; a factor beta is what's left of the premium, divided by that factor's premium; a correlation is the square root
of R squared. **Unlever, then relever**: `asset_beta` takes the listed firm's debt out, `equity_beta` puts your own
in, and the project's cost of capital comes from the asset beta either way. Each line computes one stretch answer
and checks it.

In [ ]:
developer = capm(beta_from_corr(0.1, 0.70, 0.20), 0.04, 0.05)              # P3.1: 70 % volatile, but little of it market risk
utility = capm(beta_from_corr(0.6, 0.25, 0.20), 0.04, 0.05)
premium32 = (0.118 - 0.082) / (1.4 - 0.8)                                  # P3.2: the slope of the line
rf32 = 0.082 - 0.8 * premium32                                             #        its height at beta zero
b33 = (0.11 - 0.04 - 0.9 * 0.05 - 0.5 * 0.02) / 0.03                       # P3.3: what the other factors leave
ba34 = asset_beta(0.2, 1.4, 30, 70)                                        # P3.4: unlever the listed firm
be34 = equity_beta(ba34, 0.3, 50, 50)                                      #        relever at half debt
need = [capm(b, 0.04, 0.05) for b in (0.5, 1.8)]                           # P3.5: each project against its own rate
corr36 = math.sqrt(0.49)                                                   # P3.6: the stock rises with the market
b36, specific36 = corr36 * 9 / 5, 9 * math.sqrt(1 - 0.49)
low, high = 0.9 - 2 * 0.3, 0.9 + 2 * 0.3                                   # P3.7
se_average = 0.3 / math.sqrt(16)
pv38 = 2_000_000 / 1.12 ** 3                                               # P3.8
ce38 = certainty_equivalent(2_000_000, 0.12, 0.04, 3)
early38 = certainty_equivalent(2_000_000, 0.12, 0.04, 1) / 1.04 ** 3       #        one year's risk deduction only

print(f"the developer {developer:.2%}, the utility {utility:.2%}   premium {premium32:.2%}, risk-free {rf32:.2%}   "
      f"book-to-market beta {b33:.2f}")
print(f"asset beta {ba34:.2f}, your equity beta {be34:.2f}, the project needs {capm(ba34, 0.04, 0.05):.2%}   "
      f"projects need {need[0]:.2%} and {need[1]:.2%}")
print(f"beta {b36:.2f}, specific sd {specific36:.2f} % a month   beta between {low:.1f} and {high:.1f}: cost of equity "
      f"{capm(low, 0.04, 0.05):.1%} to {capm(high, 0.04, 0.05):.1%}; sixteen firms' average ± {se_average:.3f}")
print(f"PV {pv38 / 1e6:.4f}, certainty equivalent {ce38 / 1e6:.4f}, PV if the risk is settled in year one "
      f"{early38 / 1e6:.4f} ($ millions)")
for k, v in (("P3.1", [100 * developer, 100 * utility]), ("P3.2", [100 * premium32, 100 * rf32]), ("P3.3", [b33]),
             ("P3.4", [ba34, be34, 100 * capm(ba34, 0.04, 0.05)]), ("P3.5", [100 * r for r in need]),
             ("P3.6", [b36, specific36]),
             ("P3.7", [low, high, 100 * capm(low, 0.04, 0.05), 100 * capm(high, 0.04, 0.05), se_average]),
             ("P3.8", [pv38 / 1e6, ce38 / 1e6, early38 / 1e6])):
    assert agrees(k, v), k
assert developer < utility and abs(ce38 / 1.04 ** 3 - pv38) < 1e-6 and early38 > pv38

## Tier 4 · The case: what must Bangalore Cold Chain's money earn?

Bangalore Cold Chain runs refrigerated warehouses (a fictional company; our own data). Its board can buy a
frozen-grocery home-delivery business for $40 million that should bring in $4 million a year, for good: a 10 %
return. The company's cost of capital is 9 %. **What return must the company's money earn, and what return must this
project earn?**

| what we know | |
|---|---|
| the shares | sixty months of returns for the share and for the market, in percent |
| the market | risk-free rate 4 %, market risk premium 5 % |
| the financing | debt worth $80 million (beta 0.2); 32 million shares at $10; tax 25 % |
| the project | $40 million today for an expected $4 million a year |
| three listed delivery firms | equity betas 1.8, 1.7 and 2.0; debt 25 %, 10 % and 25 % of their value (debt beta 0.2) |

**The sixty monthly returns are built in code for this example: they are not market data.** They are built so the
fitted beta comes out at 1.20. The next cell holds them (the same sixty rows as the spreadsheet's Case study tab),
builds the two tables, and checks a few of their numbers.

In [ ]:
RISK_FREE, PREMIUM = 0.04, 0.05
DEBT, DEBT_BETA = 80.0, 0.2                                                # $ millions, at market value
SHARES, PRICE = 32.0, 10.0                                                 # millions of shares; dollars a share
TAX = 0.25
PROJECT_COST, PROJECT_CASH = 40.0, 4.0                                     # $ millions: the price; the cash a year, for good
COMP_DEBT_BETA = 0.2
T58 = 2.0017                                                               # t for a 95 % range with 58 degrees of freedom

# @CASE
# monthly returns in percent, months 1 to 60: BUILT in code for this example (not market data)
MARKET = [3.83, -6.99, -0.27, -6.07, 12.13, -0.18, 0.98, 19.69, -9.03, 5.3, 1.15, 3.21, -3.1, -6.34, 7.81, 2.54,
          -3.51, 10.38, -0.41, 0.81, 2.17, 7.78, 4.49, -0.02, -5.14, 7.04, 2.43, -1.47, -2.34, 0.59, -2.49, -0.55,
          -8.27, 6.11, 3.75, -1.81, -1.49, 9.99, -7.13, -5.7, 0.7, 2.41, 6.79, -1.89, -3.43, 4.39, -11.37, 2.56, -6.6,
          -2.51, 0.42, 8.82, -1.49, -0.55, 7.57, -0.43, 5.41, 1.06, -8.4, 1.66]
SHARE = [9.19, -7.42, -14.9, -4.93, 19.46, -3.88, -7, 16.67, -23.51, 14.12, 9.72, -2.45, -19.34, -1.4, 31.28, 3.55,
         3.51, -0.8, -4.75, 14.65, -2.29, 23.23, -10.6, -0.84, 0.75, 2.85, 26.41, -6.92, -3.17, 4.25, -10.47, 0.16,
         -19.1, -1.21, 8.81, -0.43, -20.71, 5.95, 1.89, 9.01, -0.08, 5.4, 26.18, -7.57, -12.49, 7.63, -7.42, -0.87,
         -3.83, -0.61, -13.87, 12.5, -9.89, 8.6, 6.96, 6.57, 10.23, -1.71, -5.53, 0.44]
# @END CASE
returns = pd.DataFrame({"month": range(1, len(MARKET) + 1), "market": MARKET, "share": SHARE})
comps = pd.DataFrame({"firm": ["a parcel courier", "a grocery delivery firm", "a meal delivery firm"],
                      "equity_beta": [1.8, 1.7, 2.0], "debt_share": [0.25, 0.10, 0.25]})

# The case's numbers as the course computed them, to check these tables against.
# @STATS
CASE_STATS = {'beta': 1.1999675255370585,
 'a': 0.0998910171014622,
 'rsq': 0.3599618448843461,
 'corr': 0.5999682032277596,
 'sd_m': 5.773271695793151,
 'sd_s': 11.546842838976488,
 'sd_m_year': 19.999199806026137,
 'sd_s_year': 39.999436928240264,
 'steyx': 9.317045066620596,
 'se': 0.2101020036293217,
 'lo': 0.7794063448722452,
 'hi': 1.6205287062018718,
 're': 0.1,
 're_lo': 0.07897031724361227,
 're_hi': 0.1210264353100936,
 'rd': 0.05,
 'd': 80.0,
 'e': 320.0,
 'v': 400.0,
 'dv': 0.2,
 'ba': 1.0,
 'ra': 0.09,
 'wacc': 0.0875,
 'wacc_30': 0.087,
 'comp_ba': [1.4000000000000001, 1.55, 1.55],
 'proj_ba': 1.5,
 'comp_be_avg': 1.8333333333333333,
 'proj_r': 0.11500000000000002,
 'proj_r_equity': 0.13166666666666668,
 'proj_return': 0.1,
 'pv_company': 44.44444444444444,
 'pv_wacc': 45.714285714285715,
 'pv_own': 34.782608695652165,
 'npv_company': 4.444444444444443,
 'npv_wacc': 5.714285714285715,
 'npv_own': -5.217391304347835,
 'even_beta': 1.2,
 'even_cash': 4.6000000000000005,
 'borrow_rate': 0.05}
# @END STATS
S = CASE_STATS
assert len(returns) == 60 and abs(returns["market"].std() - S["sd_m"]) < 1e-9 and abs(returns["share"].std() - S["sd_s"]) < 1e-9
print(f"{len(returns)} months, {returns.shape[1]} columns: the same built returns as the course's case.")
returns.head(8)

### How you'd do this in Python

With the returns in a DataFrame, each step of the case is a line or two. The cell below reproduces the video: the
equity beta from the sixty months, the cost of equity by the CAPM, the company's own cost of capital and WACC, then
the three delivery firms unlevered and averaged to give the project a rate of its own.

In [ ]:
beta, intercept, r2, typical, se = fit_line(returns["market"], returns["share"])            # 1 · the equity beta
corr = returns["market"].corr(returns["share"])
sd_market, sd_share = returns["market"].std(), returns["share"].std()                       # pandas divides by n − 1, like STDEV.S
beta_2dp = round(beta, 2)                                                                   # the 1.20 the case carries on with
r_equity = capm(beta_2dp, RISK_FREE, PREMIUM)                                               # 2 · the cost of equity
r_debt = capm(DEBT_BETA, RISK_FREE, PREMIUM)                                                # 3 · the company
EQUITY = SHARES * PRICE
beta_assets = asset_beta(DEBT_BETA, beta_2dp, DEBT, EQUITY)
r_company = company_cost(r_debt, r_equity, DEBT, EQUITY)
after_tax_wacc = wacc(r_debt, r_equity, DEBT, EQUITY, TAX)
comps["asset_beta"] = comps["debt_share"] * COMP_DEBT_BETA + (1 - comps["debt_share"]) * comps["equity_beta"]   # 4 · the project
beta_project = comps["asset_beta"].mean()
r_project = capm(beta_project, RISK_FREE, PREMIUM)
worth_own, worth_company = PROJECT_CASH / r_project, PROJECT_CASH / r_company               # 5 · in dollars

print(f"1 · beta {beta:.4f} (intercept {intercept:.2f} % a month), R squared {r2:.4f}, correlation {corr:.2f}; "
      f"standard error {se:.2f}")
print(f"2 · cost of equity: 4 % + {beta_2dp:.2f} × 5 % = {r_equity:.2%}")
print(f"3 · debt {DEBT:.0f} + equity {EQUITY:.0f} = {DEBT + EQUITY:.0f} ($ millions); asset beta {beta_assets:.2f}; "
      f"company cost of capital {r_company:.2%}; after-tax WACC {after_tax_wacc:.2%}")
print("4 · the delivery firms' asset betas: " + ", ".join(f"{v:.2f}" for v in comps["asset_beta"])
      + f"; average {beta_project:.2f}; the project needs {r_project:.2%} and offers {PROJECT_CASH / PROJECT_COST:.0%}")
print(f"5 · worth {worth_own:.2f} at its own rate: NPV {worth_own - PROJECT_COST:+.2f}; at the company's "
      f"{r_company:.0%} it would have looked like {worth_company - PROJECT_COST:+.2f} ($ millions)".replace("-", "−"))
assert abs(beta - S["beta"]) < 1e-9 and abs(intercept - S["a"]) < 1e-9 and abs(r2 - S["rsq"]) < 1e-9
assert abs(corr - S["corr"]) < 1e-9 and abs(typical - S["steyx"]) < 1e-9 and abs(se - S["se"]) < 1e-9
assert beta_2dp == 1.2 and abs(r_equity - S["re"]) < 1e-12 and abs(r_debt - S["rd"]) < 1e-12 and EQUITY == S["e"]
assert abs(beta_assets - S["ba"]) < 1e-12 and abs(r_company - S["ra"]) < 1e-12 and abs(after_tax_wacc - S["wacc"]) < 1e-12
assert all(abs(a - b) < 1e-12 for a, b in zip(comps["asset_beta"], S["comp_ba"])) and abs(r_project - S["proj_r"]) < 1e-12
assert [round(v, 2) for v in (beta, r2, se, worth_own, worth_own - PROJECT_COST, worth_company - PROJECT_COST)] == \
       [1.2, 0.36, 0.21, 34.78, -5.22, 4.44]

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.9))
fig.subplots_adjust(left=0.08, right=0.98, top=0.85, bottom=0.2)
xs = np.array([returns["market"].min(), returns["market"].max()])
ax.axhline(0, color=DIM, lw=1); ax.axvline(0, color=DIM, lw=1)
ax.scatter(returns["market"], returns["share"], s=36, color=CHALK, alpha=0.85, zorder=3)
ax.plot(xs, intercept + beta * xs, color=GOLD, zorder=4)
ax.set_xlabel("the market's return in a month (%)"); ax.set_ylabel("the share's return (%)")
fig.suptitle(f"Sixty months: the slope of the line is beta, {beta:.2f}; the market explains {100 * r2:.0f} % of the "
             "share's variance", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · returns built in code, not market data"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: how sure the beta is and what that does to the cost of equity,
what a different tax rate does to the WACC, the project's value at three different rates, and your own
recommendation with the numbers that anchor it. Use the DataFrames `returns` and `comps` and the functions above in a
cell of your own (press **+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrames

Try the questions first. This cell works every tier 4 answer out from `returns`, `comps` and the functions above and
checks it against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
mine = {}
mine["P4.1"] = [beta, intercept]
mine["P4.2"] = [r2, corr, corr * sd_share / sd_market]                                      # the slope again
low_beta, high_beta = beta - T58 * se, beta + T58 * se
mine["P4.3"] = [typical, se, low_beta, high_beta]
mine["P4.4"] = [100 * r_equity, 100 * capm(low_beta, RISK_FREE, PREMIUM), 100 * capm(high_beta, RISK_FREE, PREMIUM)]
mine["P4.5"] = [100 * r_debt, beta_assets, 100 * r_company]
mine["P4.6"] = [100 * after_tax_wacc, 100 * wacc(r_debt, r_equity, DEBT, EQUITY, 0.30)]
mine["P4.7"] = [beta_project, 100 * r_project, PROJECT_CASH / r_company - PROJECT_COST,
                PROJECT_CASH / after_tax_wacc - PROJECT_COST, PROJECT_CASH / r_project - PROJECT_COST]
break_even_beta = (PROJECT_CASH / PROJECT_COST - RISK_FREE) / PREMIUM                       # the beta a 10 % return can carry
mine["P4.8"] = [break_even_beta, worth_own, PROJECT_COST * r_project]

for k, got in mine.items():
    got = [float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(v, 4))}" for p, v in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrames matches the practice set.")
print(f"Don't pay ${PROJECT_COST:.0f} million: at the {r_project:.1%} its risk calls for, the business is worth "
      f"{worth_own:.2f} million. It would break even only with an asset beta of {break_even_beta:.1f} (the three "
      f"delivery firms are at {', '.join(f'{v:.2f}' for v in comps['asset_beta'])}) or with "
      f"{PROJECT_COST * r_project:.1f} million of cash a year.")

## Sources
Every exercise, example and company here is our own (Bangalore Cold Chain is fictional, and its sixty monthly
returns are built in code for the example: they are not market data). The order of topics follows the risk-and-return
and cost-of-capital chapters of a standard corporate finance course; the formulas are the standard ones for beta,
the capital asset pricing model, factor models, unlevering and relevering a beta, the weighted-average cost of
capital and certainty equivalents. The spreadsheet functions named in the hints (SLOPE, INTERCEPT, RSQ, STEYX,
DEVSQ, CORREL, STDEV.S, SUMPRODUCT, AVERAGE, SQRT) have the same names and take their arguments in the same order in
the two most widely used spreadsheets (their official function lists, retrieved 2026-11-11).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/cf05_capm_cost_of_capital/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("cf05_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["group"]) == \
               (_e["q"], _e["working"], _e["function"], _e["tier"], _e["group"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) == len(_q["parts"]) and all(
            abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        _t = _e["tol"] if isinstance(_e["tol"], tuple) else (_e["tol"],) * len(_q["answer"])
        assert list(_t) == _q["tol"], _e["id"]
    assert len(_P.EX) == len(EX)
    assert set(_P.CS) == set(CASE_STATS) and all(
        abs(float(a) - b) < 1e-9 for k in CASE_STATS for a, b in zip(_flat(_P.CS[k]), _flat(CASE_STATS[k])))
    assert _P.CASE["market"] == MARKET and _P.CASE["share"] == SHARE and _P.T58 == T58
    assert (_P.CASE["rf"], _P.CASE["prem"], _P.CASE["debt_m"], _P.CASE["debt_beta"], _P.CASE["tax"]) == \
           (RISK_FREE, PREMIUM, DEBT, DEBT_BETA, TAX)
    assert (_P.FACTS["shares_m"], _P.FACTS["share_price"], _P.CASE["equity_m"]) == (SHARES, PRICE, SHARES * PRICE)
    assert _P.CASE["project"] == {"cost_m": PROJECT_COST, "cash_m": PROJECT_CASH} and _P.CASE["comp_debt_beta"] == COMP_DEBT_BETA
    assert [tuple(c) for c in _P.CASE["comps"]] == list(zip(comps["firm"], comps["equity_beta"], comps["debt_share"]))
    assert (_P.SIX_M, _P.SIX_S) == (market.tolist(), stock.tolist())
    assert (_P.FIRM["bd"], _P.FIRM["be"], _P.FIRM["d"], _P.FIRM["e"], _P.FIRM["tax"]) == (0.4, 1.6, 30.0, 70.0, 0.25)
    _sheet = pathlib.Path("../../../sheets/src/corporate_finance/capm_cost_of_capital.py")
    if _sheet.exists():                                                    # the spreadsheet's part labels are these
        _spec = importlib.util.spec_from_file_location("cf05_sheet", _sheet)
        _S = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_S)
        assert all(_S.PARTS[k] == Q[k]["parts"] and _S.PCT.get(k, []) == Q[k]["pct"] for k in Q)
    print("matches the course's practice set")